# POI Ingestion - Bronze Layer

Downloads a Geofabrik OSM extract for the expansion state, parses it locally with osmium to extract partner and competitor brand locations.

**Data Source:** Geofabrik OSM PBF extract → parsed with osmium for branded POIs

**Approach:**
- Download state-level PBF extract to Databricks volume (cached for re-runs)
- Parse locally with osmium to find POIs matching brand names from config
- General POI counts (retail, food_drink, etc.) come from CARTO Marketplace at H3 level

**Brand Configuration:** Loaded from `poi_config.yml` (single source of truth)

**Output Table:**
- `{catalog}.{bronze_schema}.raw_pois` - Branded POI data with tags

## Parameters

In [ ]:
import requests
import time
import yaml
import os
import shutil
import osmium
from pyspark.sql import functions as F
from pyspark.sql.types import *
from collections import Counter

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("config_path", "")
dbutils.widgets.text("expansion_state", "MA")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
config_path = dbutils.widgets.get("config_path")
expansion_state = dbutils.widgets.get("expansion_state")

assert catalog and bronze_schema and config_path, "Missing required parameters: catalog, bronze_schema, config_path"

output_table = f"{catalog}.{bronze_schema}.raw_pois"

# OSM extract URL for state
# Using Geofabrik state extracts (much smaller than full US)
OSM_EXTRACTS = {
    "MA": "https://download.geofabrik.de/north-america/us/massachusetts-latest.osm.pbf",
    "MI": "https://download.geofabrik.de/north-america/us/michigan-latest.osm.pbf",
    "VA": "https://download.geofabrik.de/north-america/us/virginia-latest.osm.pbf",
    "NY": "https://download.geofabrik.de/north-america/us/new-york-latest.osm.pbf",
    "WA": "https://download.geofabrik.de/north-america/us/washington-latest.osm.pbf",
    "MD": "https://download.geofabrik.de/north-america/us/maryland-latest.osm.pbf",
    "NJ": "https://download.geofabrik.de/north-america/us/new-jersey-latest.osm.pbf",
}

osm_url = OSM_EXTRACTS.get(expansion_state)
if not osm_url:
    raise ValueError(f"No OSM extract URL for state: {expansion_state}")

print(f"Catalog: {catalog}")
print(f"Schema: {bronze_schema}")
print(f"Config: {config_path}")
print(f"Expansion state: {expansion_state}")
print(f"OSM extract: {osm_url}")
print(f"Output table: {output_table}")

## Load Brand Configuration

In [ ]:
# Load brand configuration from poi_config.yml (single source of truth)
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

brands_config = config.get('brands', {})
partner_brands = brands_config.get('partner_brands', [])
competitor_brands = brands_config.get('competitor_brands', [])

assert partner_brands, "No partner_brands defined in poi_config.yml"
assert competitor_brands, "No competitor_brands defined in poi_config.yml"

all_brands = partner_brands + competitor_brands

print(f"Partner brands ({len(partner_brands)}): {partner_brands}")
print(f"Competitor brands ({len(competitor_brands)}): {competitor_brands}")
print(f"Total brands to search: {len(all_brands)}")

## Download & Parse OSM Extract

In [ ]:
# Download OSM extract to Databricks volume
osm_filename = osm_url.split('/')[-1]
osm_volume_path = f"/Volumes/{catalog}/{bronze_schema}/osm_data/"
osm_file_path = f"{osm_volume_path}{osm_filename}"

print(f"Checking for OSM extract: {osm_file_path}")

# Ensure volume directory exists
try:
    dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_schema}/osm_data/")
except:
    pass

# Check if file already exists (idempotency)
if os.path.exists(osm_file_path):
    file_size_mb = os.path.getsize(osm_file_path) / (1024 * 1024)
    print(f"✓ File already exists: {osm_file_path} ({file_size_mb:.1f} MB)")
    print(f"  Skipping download (delete file to re-download)")
else:
    print(f"Downloading OSM extract from {osm_url}...")
    print(f"  This may take 2-5 minutes depending on state size...")
    
    start_time = time.time()
    
    # Stream download directly to volume (memory efficient)
    with requests.get(osm_url, stream=True, timeout=600, 
                     headers={"User-Agent": "DatabricksGeospatialPipeline/1.0"}) as r:
        r.raise_for_status()
        total_size = int(r.headers.get('content-length', 0))
        total_size_mb = total_size / (1024 * 1024)
        
        print(f"  File size: {total_size_mb:.1f} MB")
        
        with open(osm_file_path, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16*1024*1024)  # 16MB chunks
    
    download_time = time.time() - start_time
    print(f"  ✓ Downloaded in {download_time:.1f}s")

print(f"\n{'='*60}")
print(f"OSM Extract ready: {osm_file_path}")
print(f"{'='*60}")

In [ ]:
# Parse OSM extract for branded POIs
print(f"\n{'='*60}")
print(f"Parsing OSM extract for branded POIs...")
print(f"{'='*60}")

class POIHandler(osmium.SimpleHandler):
    """OSM handler that extracts branded POI nodes and ways."""
    
    def __init__(self, brand_names):
        super().__init__()
        self.pois = []
        # Use a set of lowercased brand names for exact matching
        self.brand_names = {b.lower().strip() for b in brand_names}
        self.poi_categories = ['amenity', 'shop']  # Only look at amenity/shop tags
    
    def _matches_brand(self, tags):
        """Check if POI name exactly matches any brand name."""
        name = tags.get('name', '').lower().strip()
        if not name:
            return False
        return name in self.brand_names
    
    def _has_poi_tag(self, tags):
        """Check if element has a relevant POI tag."""
        return any(category in tags for category in self.poi_categories)
    
    def node(self, n):
        """Extract branded POI nodes with coordinates."""
        if self._has_poi_tag(n.tags) and self._matches_brand(n.tags):
            if n.location.valid():
                self.pois.append({
                    'osm_id': str(n.id),
                    'osm_type': 'node',
                    'latitude': n.location.lat,
                    'longitude': n.location.lon,
                    'tags': dict(n.tags)
                })
    
    def way(self, w):
        """Extract branded POI ways (buildings) with centroid."""
        if self._has_poi_tag(w.tags) and self._matches_brand(w.tags):
            lats, lons = [], []
            for node in w.nodes:
                if node.location.valid():
                    lats.append(node.location.lat)
                    lons.append(node.location.lon)
            
            if lats and lons:
                self.pois.append({
                    'osm_id': str(w.id),
                    'osm_type': 'way',
                    'latitude': sum(lats) / len(lats),  # Centroid
                    'longitude': sum(lons) / len(lons),
                    'tags': dict(w.tags)
                })

# Get all brand names from config
all_brands = partner_brands + competitor_brands

print(f"Searching for {len(all_brands)} exact brand names:")
for brand in all_brands:
    print(f"  - \"{brand}\"")

# Parse OSM file
start_time = time.time()
handler = POIHandler(brand_names=all_brands)
handler.apply_file(osm_file_path, locations=True)  # locations=True resolves node coordinates for ways
parse_time = time.time() - start_time

print(f"\n✓ Parsed {len(handler.pois)} branded POIs in {parse_time:.1f}s")

# Count by brand
brand_counts = Counter()
for poi in handler.pois:
    name = poi['tags'].get('name', 'Unknown')
    brand_counts[name] += 1

print(f"\nPOIs by brand:")
for brand, count in brand_counts.most_common(20):
    print(f"  {brand}: {count}")

if len(handler.pois) == 0:
    raise RuntimeError(
        f"No branded POIs found in {expansion_state}. "
        f"Check brand names in poi_config.yml"
    )

# Store results for next cell
pois = handler.pois

## Write to Bronze Table

In [ ]:
# Convert POIs to Spark DataFrame (same schema as original raw_pois)
schema = StructType([
    StructField("osm_id", StringType(), False),
    StructField("osm_type", StringType(), False),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("tags", MapType(StringType(), StringType()), True)
])

poi_df = spark.createDataFrame(pois, schema=schema)
poi_df = poi_df.withColumn("ingestion_timestamp", F.current_timestamp())

print(f"Created DataFrame with {poi_df.count()} rows")
display(poi_df.limit(10))

In [ ]:
# Write to Bronze table
(poi_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(output_table))

print(f"✓ Written {poi_df.count()} POIs to {output_table}")

## Validation

In [ ]:
print("=" * 80)
print("POI INGESTION VALIDATION")
print("=" * 80)

summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_pois,
        COUNT(DISTINCT osm_id) as unique_pois,
        COUNT(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 END) as pois_with_coords,
        COUNT(CASE WHEN tags['name'] IS NOT NULL THEN 1 END) as pois_with_name,
        COUNT(CASE WHEN tags['addr:street'] IS NOT NULL THEN 1 END) as pois_with_address
    FROM {output_table}
""")
display(summary)

# Show all extracted branded POIs
print("\nExtracted branded POIs:")
spark.sql(f"""
    SELECT 
        tags['name'] as name,
        COALESCE(tags['amenity'], tags['shop']) as category,
        osm_type,
        COUNT(*) as count
    FROM {output_table}
    WHERE tags['name'] IS NOT NULL
    GROUP BY tags['name'], COALESCE(tags['amenity'], tags['shop']), osm_type
    ORDER BY count DESC
""").show(50, truncate=False)

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)